Urban Data Science & Smart Cities <br>
URSP688Y Spring 2026<br>
Instructor: Chester Harvey <br>
Urban Studies & Planning <br>
National Center for Smart Growth <br>
University of Maryland

The files needed for this analysis are easy to download. The code is oriented so each of the files can be located through the file path itself. The necessary files are metro_boundaries.json, Levees.json, County.json, LODES_json, and NFHL. The hardest part will be grabbing the metro area codes, to do this, you can either google it or do control f to find the name of the metro within metro boudndaries and the numerical code is adjacent. The code utilizes these datasets to create an analysis of floodplain and levee development within the American Midwest and Upper South. The deliverable folders are the CSV_Levee, CSV_Sponsor, and CSV. The last CSV is for the floodplain development. I simlpy recombined these into a excel using the grab data from file within excel, which created pivotable tables which are the source of the final work. 

In [1]:
import pandas as pd
import geopandas as gpd

In [ ]:
#Importing National County Boundaries
county = gpd.read_file('County.json')
county = county.to_crs(6933)

In [2]:
#Importing metro areas
metros = gpd.read_file('metro_boundaries.json')
metros = metros.to_crs(6933)

In [ ]:
#This function gets the number of levee protected jobs by metro area
def levee_function(LODES, metro):
    #creates geopandas
    levees = gpd.read_file('Levees.json')
    LODES = gpd.read_file(LODES)
    #Converts to WGS84
    LODES = LODES.to_crs(6933)
    levees = levees.to_crs(6933)
    metro = metro.to_crs(6933)
    #Clips the levees
    levees_clip = gpd.clip(levees, metro)
    #spatially joins the points to levees
    join = LODES.sjoin(levees_clip, how = "left")
    #seperates the data into points inside or outside a levee system
    join_y = join[~join['LEVEED_ID'].isna()]
    join_n = join[join['LEVEED_ID'].isna()]
    #Takes only quant variables, used to create two new dataframes summarizing data
    list = join_y.columns[2:43].tolist()
    data_y = join_y[list].sum()
    data_n = join_n[list].sum()
    #Turns the series into dataframes
    data_y = pd.DataFrame(data_y)
    data_n = pd.DataFrame(data_n)
    #Reorganizes the columns
    data_y = data_y.reset_index().rename(columns={data_y.columns[0]:'data'})
    data_n = data_n.reset_index().rename(columns={data_n.columns[0]:'data'})
    #Merges the two dataframes
    data = pd.merge(data_y, data_n, on = 'index')
    #Renames columns for readability
    data['levee'] = data['data_x']
    data['non_levee'] = data['data_y']
    #Removes unncessary columns
    data = data.drop('data_x', axis=1)
    data = data.drop('data_y', axis=1)
    #Creates a new column which indicates what percent of the category are within a levee
    data['pct_levee'] = data['levee'] / (data['levee'] + data['non_levee'])
    return data

In [ ]:
def levee_function_overtop(LODES, metro):
    #creates geopandas
    levees = gpd.read_file('Levees.json')
    LODES = gpd.read_file(LODES)
    #Converts to WGS84
    LODES = LODES.to_crs(6933)
    levees = levees.to_crs(6933)
    metro = metro.to_crs(6933)
    #Clips the levees
    levees_clip = gpd.clip(levees, metro)
    #spatially joins the points to levees
    join = LODES.sjoin(levees_clip, how = "left")
    #seperates the data into points inside or outside a levee system
    join_y = join[~join['LEVEED_ID'].isna()]
    #Calculates the floodplain area per county
    pivot = join_y.pivot_table(index='OVERTOPPING_ACE', values='c000', aggfunc='sum')
    return pivot

In [ ]:
#This function creates the data for levee sponsor type
def levee_function_sponsor(LODES, metro):
    #creates geopandas
    levees = gpd.read_file('Levees.json')
    LODES = gpd.read_file(LODES)
    #Converts to WGS84
    LODES = LODES.to_crs(6933)
    levees = levees.to_crs(6933)
    metro = metro.to_crs(6933)
    #Clips the levees
    levees_clip = gpd.clip(levees, metro)
    #spatially joins the points to levees
    join = LODES.sjoin(levees_clip, how = "left")
    #seperates the data into points inside or outside a levee system
    join_y = join[~join['LEVEED_ID'].isna()]
    #Calculates the floodplain area per county
    pivot = join_y.pivot_table(index='SPONSOR_TYPE', values='c000', aggfunc='sum')
    return pivot

In [ ]:
def Floodplain_Function(LODES, NFHL, metro):
    #creates geopandas
    LODES = gpd.read_file(LODES)
    NFHL = gpd.read_file(NFHL)
    #Converts to WGS84
    LODES = LODES.to_crs(6933)
    NFHL = NFHL.to_crs(6933)
    metro = metro.to_crs(6933)
    #Clips the counties based off of the metro area
    metro_counties = gpd.clip(county, metro)
    #Clips floodplains based off exact metro boundaries
    NFHL = gpd.clip(NFHL, metro)
    #Calculate square miles of NFHL
    area = NFHL.geometry.area
    NFHL['sqmi'] = area / 2589988.11
    #Calculates the centroids of floodplain polygons
    NFHL['centroid'] = NFHL.geometry.centroid
    NFHL_cntr = NFHL.set_geometry('centroid')
    cnty_join = gpd.sjoin(NFHL_cntr, metro_counties, how='left', predicate='within')
    #Calculates the floodplain area per county
    Floodplain_Functionlood_area = cnty_join.pivot_table(index='GISJOIN', values='sqmi', aggfunc='sum')
    #Joins countie to floodplain area per county
    county_data = metro_counties.merge(Floodplain_Functionlood_area,on="GISJOIN",how="left")
    #Filters by counties which have data
    counties = county_data[(county_data['sqmi'] > 1) & (~county_data['sqmi'].isna())]
    #Clips LODES data by counties which have digitized floodplain data
    LODES = gpd.clip(LODES, counties)
    #spatially joins the points to levees
    join = LODES.sjoin(NFHL, how = "left")
    #seperates the data into points inside or outside a levee system
    join_y = join[~join['DFIRM_ID'].isna()]
    join_n = join[join['DFIRM_ID'].isna()]
    #Takes only quant variables, used to create two new dataframes summarizing data
    list = join_y.columns[2:43].tolist()
    data_y = join_y[list].sum()
    data_n = join_n[list].sum()
    #Turns the series into dataframes
    data_y = pd.DataFrame(data_y)
    data_n = pd.DataFrame(data_n)
    #Reorganizes the columns
    data_y = data_y.reset_index().rename(columns={data_y.columns[0]:'data'})
    data_n = data_n.reset_index().rename(columns={data_n.columns[0]:'data'})
    #Merges the two dataframes
    data = pd.merge(data_y, data_n, on = 'index')
    #Renames columns for readability
    data['floodplain'] = data['data_x']
    data['non_floodplain'] = data['data_y']
    #Removes unncessary columns
    data = data.drop('data_x', axis=1)
    data = data.drop('data_y', axis=1)
    #Creates a new column which indicates what percent of the category are within a levee
    data['pct_floodplain'] = data['floodplain'] / (data['floodplain'] + data['non_floodplain'])
    return data

In [ ]:
#Does the function to find the levee sponsor types for each metro area

#Oklahoma City Levee
OKC_Metro = metros[metros['GEOID'] == '36420'].copy()
OKC_Work = levee_function_sponsor('LODES_json/OKC_W.json', OKC_Metro)
OKC_Work.to_csv('CSV_sponsor/OKC_Levee.csv')
#Milwaukee Levee
MLW_Metro = metros[metros['GEOID'] == '33340'].copy()
MLW_Work = levee_function_sponsor('LODES_json/MLW_W.json', MLW_Metro)
MLW_Work.to_csv('CSV_sponsor/MLW_Levee.csv')
#Little Rock
LTR_Metro = metros[metros['GEOID'] == '30780'].copy()
LTR_Work = levee_function_sponsor('LODES_json/LTR_W.json', LTR_Metro)
LTR_Work.to_csv('CSV_sponsor/LTR_Levee.csv')
#Nashville
NSV_Metro = metros[metros['GEOID'] == '34980'].copy()
NSV_Work = levee_function_sponsor('LODES_json/NSV_W.json', NSV_Metro)
NSV_Work.to_csv('CSV_sponsor/NSV_Levee.csv')
#St. Louis
STL_Metro = metros[metros['GEOID'] == '41180'].copy()
STL_Work = levee_function_sponsor('LODES_json/STL_W.json', STL_Metro)
STL_Work.to_csv('CSV_sponsor/STL_Levee.csv')
#Kansas City
KC_Metro = metros[metros['GEOID'] == '28140'].copy()
KC_Work = levee_function_sponsor('LODES_json/KC_W.json', KC_Metro)
KC_Work.to_csv('CSV_sponsor/KC_Levee.csv')
#Memphis
Mem_Metro = metros[metros['GEOID'] == '32820'].copy()
Mem_Work = levee_function_sponsor('LODES_json/MEM_W.json', Mem_Metro)
Mem_Work.to_csv('CSV_sponsor/Mem_Levee.csv')
#Chicago
CHC_Metro = metros[metros['GEOID'] == '16980'].copy()
CHC_Work = levee_function_sponsor('LODES_json/CHC_W.json', CHC_Metro)
CHC_Work.to_csv('CSV_sponsor/CHC_Levee.csv')
#Columbus
CLM_Metro = metros[metros['GEOID'] == '18140'].copy()
CLM_Work = levee_function_sponsor('LODES_json/CLM_W.json', CLM_Metro)
CLM_Work.to_csv('CSV_sponsor/CLM_Levee.csv')
#Cleveland
CLV_Metro = metros[metros['GEOID'] == '17460'].copy()
CLV_Work = levee_function_sponsor('LODES_json/CLV_W.json', CLV_Metro)
CLV_Work.to_csv('CSV_sponsor/CLV_Levee.csv')
#Cincinnati
CNC_Metro = metros[metros['GEOID'] == '17140'].copy()
CNC_Work = levee_function_sponsor('LODES_json/CNC_W.json', CNC_Metro)
CNC_Work.to_csv('CSV_sponsor/CNC_Levee.csv')
#Des Moines
DSM_Metro = metros[metros['GEOID'] == '19780'].copy()
DSM_Work = levee_function_sponsor('LODES_json/DSM_W.json', DSM_Metro)
DSM_Work.to_csv('CSV_sponsor/DSM_Levee.csv')
#Indianapolis
IND_Metro = metros[metros['GEOID'] == '26900'].copy()
IND_Work = levee_function_sponsor('LODES_json/IND_W.json', IND_Metro)
IND_Work.to_csv('CSV_sponsor/IND_Levee.csv')
#Louisville
LSV_Metro = metros[metros['GEOID'] == '31140'].copy()
LSV_Work = levee_function_sponsor('LODES_json/LSV_W.json', LSV_Metro)
LSV_Work.to_csv('CSV_sponsor/LSV_Levee.csv')
#Minneapolis
MIN_Metro = metros[metros['GEOID'] == '33460'].copy()
MIN_Work = levee_function_sponsor('LODES_json/MIN_W.json', MIN_Metro)
MIN_Work.to_csv('CSV_sponsor/MIN_Levee.csv')
#Omaha
OMA_Metro = metros[metros['GEOID'] == '36540'].copy()
OMA_Work = levee_function_sponsor('LODES_json/OMA_W.json', OMA_Metro)
OMA_Work.to_csv('CSV_sponsor/OMA_Levee.csv')

In [ ]:
#Gets the total number of jobs which are levee protected by metro area
#Oklahoma City Levee
OKC_Metro = metros[metros['GEOID'] == '36420'].copy()
OKC_Work = levee_function('LODES_json/OKC_W.json', OKC_Metro)
OKC_Home = levee_function('LODES_json/OKC_H.json', OKC_Metro)
OKC = OKC_Work.merge(OKC_Home,on="index",how="left")
OKC.to_csv('CSV_Levee/OKC_Levee.csv')
#Milwaukee Levee
MLW_Metro = metros[metros['GEOID'] == '33340'].copy()
MLW_Work = levee_function('LODES_json/MLW_W.json', MLW_Metro)
MLW_Home = levee_function('LODES_json/MLW_H.json', MLW_Metro)
MLW = MLW_Work.merge(MLW_Home,on="index",how="left")
MLW.to_csv('CSV_Levee/MLW_Levee.csv')
#Little Rock
LTR_Metro = metros[metros['GEOID'] == '30780'].copy()
LTR_Work = levee_function('LODES_json/LTR_W.json', LTR_Metro)
LTR_Home = levee_function('LODES_json/LTR_H.json', LTR_Metro)
LTR = LTR_Work.merge(LTR_Home,on="index",how="left")
LTR.to_csv('CSV_Levee/LTR_Levee.csv')
#Nashville
NSV_Metro = metros[metros['GEOID'] == '34980'].copy()
NSV_Work = levee_function('LODES_json/NSV_W.json', NSV_Metro)
NSV_Home = levee_function('LODES_json/NSV_H.json', NSV_Metro)
NSV = NSV_Work.merge(NSV_Home,on="index",how="left")
NSV.to_csv('CSV_Levee/NSV_Levee.csv')
#St. Louis
STL_Metro = metros[metros['GEOID'] == '41180'].copy()
STL_Work = levee_function('LODES_json/STL_W.json', STL_Metro)
STL_Home = levee_function('LODES_json/STL_H.json', STL_Metro)
STL = STL_Work.merge(STL_Home,on="index",how="left")
STL.to_csv('CSV_Levee/STL_Levee.csv')
#Kansas City
KC_Metro = metros[metros['GEOID'] == '28140'].copy()
KC_Work = levee_function('LODES_json/KC_W.json', KC_Metro)
KC_Home = levee_function('LODES_json/KC_H.json', KC_Metro)
KC = KC_Work.merge(KC_Home,on="index",how="left")
KC.to_csv('CSV_Levee/KC_Levee.csv')
#Memphis
Mem_Metro = metros[metros['GEOID'] == '32820'].copy()
Mem_Work = levee_function('LODES_json/MEM_W.json', Mem_Metro)
Mem_Home = levee_function('LODES_json/MEM_H.json', Mem_Metro)
Mem = Mem_Work.merge(Mem_Home,on="index",how="left")
Mem.to_csv('CSV_Levee/Mem_Levee.csv')
#Chicago
CHC_Metro = metros[metros['GEOID'] == '16980'].copy()
CHC_Work = levee_function('LODES_json/CHC_W.json', CHC_Metro)
CHC_Home = levee_function('LODES_json/CHC_H.json', CHC_Metro)
CHC = CHC_Work.merge(CHC_Home,on="index",how="left")
CHC.to_csv('CSV_Levee/CHC_Levee.csv')
#Columbus
CLM_Metro = metros[metros['GEOID'] == '18140'].copy()
CLM_Work = levee_function('LODES_json/CLM_W.json', CLM_Metro)
CLM_Home = levee_function('LODES_json/CLM_H.json', CLM_Metro)
CLM = CLM_Work.merge(CLM_Home,on="index",how="left")
CLM.to_csv('CSV_Levee/CLM_Levee.csv')
#Cleveland
CLV_Metro = metros[metros['GEOID'] == '17460'].copy()
CLV_Work = levee_function('LODES_json/CLV_W.json', CLV_Metro)
CLV_Home = levee_function('LODES_json/CLV_H.json', CLV_Metro)
CLV = CLV_Work.merge(CLV_Home,on="index",how="left")
CLV.to_csv('CSV_Levee/CLV_Levee.csv')
#Cincinnati
CNC_Metro = metros[metros['GEOID'] == '17140'].copy()
CNC_Work = levee_function('LODES_json/CNC_W.json', CNC_Metro)
CNC_Home = levee_function('LODES_json/CNC_H.json', CNC_Metro)
CNC = CNC_Work.merge(CNC_Home,on="index",how="left")
CNC.to_csv('CSV_Levee/CNC_Levee.csv')
#Des Moines
DSM_Metro = metros[metros['GEOID'] == '19780'].copy()
DSM_Work = levee_function('LODES_json/DSM_W.json', DSM_Metro)
DSM_Home = levee_function('LODES_json/DSM_H.json', DSM_Metro)
DSM = DSM_Work.merge(DSM_Home,on="index",how="left")
DSM.to_csv('CSV_Levee/DSM_Levee.csv')
#Indianapolis
IND_Metro = metros[metros['GEOID'] == '26900'].copy()
IND_Work = levee_function('LODES_json/IND_W.json', IND_Metro)
IND_Home = levee_function('LODES_json/IND_H.json', IND_Metro)
IND = IND_Work.merge(IND_Home,on="index",how="left")
IND.to_csv('CSV_Levee/IND_Levee.csv')
#Louisville
LSV_Metro = metros[metros['GEOID'] == '31140'].copy()
LSV_Work = levee_function('LODES_json/LSV_W.json', LSV_Metro)
LSV_Home = levee_function('LODES_json/LSV_H.json', LSV_Metro)
LSV = LSV_Work.merge(LSV_Home,on="index",how="left")
LSV.to_csv('CSV_Levee/LSV_Levee.csv')
#Minneapolis
MIN_Metro = metros[metros['GEOID'] == '33460'].copy()
MIN_Work = levee_function('LODES_json/MIN_W.json', MIN_Metro)
MIN_Home = levee_function('LODES_json/MIN_H.json', MIN_Metro)
MIN = MIN_Work.merge(MIN_Home,on="index",how="left")
MIN.to_csv('CSV_Levee/MIN_Levee.csv')
#Omaha
OMA_Metro = metros[metros['GEOID'] == '36540'].copy()
OMA_Work = levee_function('LODES_json/OMA_W.json', OMA_Metro)
OMA_Home = levee_function('LODES_json/OMA_H.json', OMA_Metro)
OMA = OMA_Work.merge(OMA_Home,on="index",how="left")
OMA.to_csv('CSV_Levee/OMA_Levee.csv')

/Users/Meggittmen/miniconda3/envs/ursp688y_sp2026_fm/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(
/Users/Meggittmen/miniconda3/envs/ursp688y_sp2026_fm/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: EPSG:102003 is not a valid CRS code, but ESRI:102003 is. Assuming ESRI:102003 was meant
  return ogr_read(


In [ ]:
#Oklahoma City
OKC_Metro = metros[metros['GEOID'] == '36420'].copy()
OKC_Work = Floodplain_Function('LODES_json/OKC_W.json', 'NFHL/OKC_NFHL.json',OKC_Metro)
OKC_Home = Floodplain_Function('LODES_json/OKC_H.json', 'NFHL/OKC_NFHL.json',OKC_Metro)
OKC = OKC_Work.merge(OKC_Home,on="index",how="left")
OKC.to_csv('CSV/OKC.csv')

In [ ]:
#Milwaukee
MLW_Metro = metros[metros['GEOID'] == '33340'].copy()
MLW_Work = Floodplain_Function('LODES_json/MLW_W.json', 'NFHL/MLW_NFHL.json', MLW_Metro)
MLW_Home = Floodplain_Function('LODES_json/MLW_H.json', 'NFHL/MLW_NFHL.json', MLW_Metro)
MLW = MLW_Work.merge(MLW_Home,on="index",how="left")
MLW.to_csv('CSV/MLW.csv')

In [ ]:
#Little Rock
LTR_Metro = metros[metros['GEOID'] == '30780'].copy()
LTR_Work = Floodplain_Function('LODES_json/LTR_W.json', 'NFHL/LTR_NFHL.json', LTR_Metro)
LTR_Home = Floodplain_Function('LODES_json/LTR_H.json', 'NFHL/LTR_NFHL.json', LTR_Metro)
LTR = LTR_Work.merge(LTR_Home,on="index",how="left")
LTR.to_csv('CSV/LTR.csv')

In [ ]:
#Nashville
NSV_Metro = metros[metros['GEOID'] == '34980'].copy()
NSV_Work = Floodplain_Function('LODES_json/NSV_W.json', 'NFHL/NSV_NFHL.json', NSV_Metro)
NSV_Home = Floodplain_Function('LODES_json/NSV_H.json', 'NFHL/NSV_NFHL.json', NSV_Metro)
NSV = NSV_Work.merge(NSV_Home,on="index",how="left")
NSV.to_csv('CSV/NSV.csv')

In [ ]:
#St. Louis
STL_Metro = metros[metros['GEOID'] == '41180'].copy()
STL_Work = Floodplain_Function('LODES_json/STL_W.json', 'NFHL/StLouis_NFHL.json', STL_Metro)
STL_Home = Floodplain_Function('LODES_json/STL_H.json', 'NFHL/StLouis_NFHL.json', STL_Metro)
STL = STL_Work.merge(STL_Home,on="index",how="left")
STL.to_csv('CSV/STL.csv')

In [ ]:
#Kansas City
KC_Metro = metros[metros['GEOID'] == '28140'].copy()
KC_Work = Floodplain_Function('LODES_json/KC_W.json', 'NFHL/KansasCity_NFHL.json', KC_Metro)
KC_Home = Floodplain_Function('LODES_json/KC_H.json', 'NFHL/KansasCity_NFHL.json', KC_Metro)
KC = KC_Work.merge(KC_Home,on="index",how="left")
KC.to_csv('CSV/KC.csv')

In [ ]:
#Memphis
Mem_Metro = metros[metros['GEOID'] == '32820'].copy()
Mem_Work = Floodplain_Function('LODES_json/MEM_W.json', 'NFHL/Memphis_NFHL.json', Mem_Metro)
Mem_Home = Floodplain_Function('LODES_json/MEM_H.json', 'NFHL/Memphis_NFHL.json', Mem_Metro)
Mem = Mem_Work.merge(Mem_Home,on="index",how="left")
Mem.to_csv('CSV/Mem.csv')

In [ ]:
#Chicago
CHC_Metro = metros[metros['GEOID'] == '16980'].copy()
CHC_Work = Floodplain_Function('LODES_json/CHC_W.json', 'NFHL/Chicago_NFHL.json', CHC_Metro)
CHC_Home = Floodplain_Function('LODES_json/CHC_H.json', 'NFHL/Chicago_NFHL.json', CHC_Metro)
CHC = CHC_Work.merge(CHC_Home,on="index",how="left")
CHC.to_csv('CSV/CHC.csv')

In [ ]:
#Columbus
CLM_Metro = metros[metros['GEOID'] == '18140'].copy()
CLM_Work = Floodplain_Function('LODES_json/CLM_W.json', 'NFHL/Columbus_NFHL.json', CLM_Metro)
CLM_Home = Floodplain_Function('LODES_json/CLM_H.json', 'NFHL/Columbus_NFHL.json', CLM_Metro)
CLM = CLM_Work.merge(CLM_Home,on="index",how="left")
CLM.to_csv('CSV/CLM.csv')

In [ ]:
#Cleveland
CLV_Metro = metros[metros['GEOID'] == '17460'].copy()
CLV_Work = Floodplain_Function('LODES_json/CLV_W.json', 'NFHL/Cleveland_NFHL.json', CLV_Metro)
CLV_Home = Floodplain_Function('LODES_json/CLV_H.json', 'NFHL/Cleveland_NFHL.json', CLV_Metro)
CLV = CLV_Work.merge(CLV_Home,on="index",how="left")
CLV.to_csv('CSV/CLV.csv')

In [ ]:
#Cincinnati
CNC_Metro = metros[metros['GEOID'] == '17140'].copy()
CNC_Work = Floodplain_Function('LODES_json/CNC_W.json', 'NFHL/Cincy_NFHL.json', CNC_Metro)
CNC_Home = Floodplain_Function('LODES_json/CNC_H.json', 'NFHL/Cincy_NFHL.json', CNC_Metro)
CNC = CNC_Work.merge(CNC_Home,on="index",how="left")
CNC.to_csv('CSV/CNC.csv')

In [ ]:
#Des Moines
DSM_Metro = metros[metros['GEOID'] == '19780'].copy()
DSM_Work = Floodplain_Function('LODES_json/DSM_W.json', 'NFHL/DesMoines_NFHL.json', DSM_Metro)
DSM_Home = Floodplain_Function('LODES_json/DSM_H.json', 'NFHL/DesMoines_NFHL.json', DSM_Metro)
DSM = DSM_Work.merge(DSM_Home,on="index",how="left")
DSM.to_csv('CSV/DSM.csv')

In [ ]:
#Indianapolis
IND_Metro = metros[metros['GEOID'] == '26900'].copy()
IND_Work = Floodplain_Function('LODES_json/IND_W.json', 'NFHL/Indianapolis_NFHL.json', IND_Metro)
IND_Home = Floodplain_Function('LODES_json/IND_H.json', 'NFHL/Indianapolis_NFHL.json', IND_Metro)
IND = IND_Work.merge(IND_Home,on="index",how="left")
IND.to_csv('CSV/IND.csv')

In [ ]:
#Louisville
LSV_Metro = metros[metros['GEOID'] == '31140'].copy()
LSV_Work = Floodplain_Function('LODES_json/LSV_W.json', 'NFHL/Louisville_NFHL.json', LSV_Metro)
LSV_Home = Floodplain_Function('LODES_json/LSV_H.json', 'NFHL/Louisville_NFHL.json', LSV_Metro)
LSV = LSV_Work.merge(LSV_Home,on="index",how="left")
LSV.to_csv('CSV/LSV.csv')

In [ ]:
#Minneapolis
MIN_Metro = metros[metros['GEOID'] == '33460'].copy()
MIN_Work = Floodplain_Function('LODES_json/MIN_W.json', 'NFHL/Minneapolis_NFHL.json', MIN_Metro)
MIN_Home = Floodplain_Function('LODES_json/MIN_H.json', 'NFHL/Minneapolis_NFHL.json', MIN_Metro)
MIN = MIN_Work.merge(MIN_Home,on="index",how="left")
MIN.to_csv('CSV/MIN.csv')

In [ ]:
#Omaha
OMA_Metro = metros[metros['GEOID'] == '36540'].copy()
OMA_Work = Floodplain_Function('LODES_json/OMA_W.json', 'NFHL/Omaha_NFHL.json', OMA_Metro)
OMA_Home = Floodplain_Function('LODES_json/OMA_H.json', 'NFHL/Omaha_NFHL.json', OMA_Metro)
OMA = OMA_Work.merge(OMA_Home,on="index",how="left")
OMA.to_csv('CSV/OMA.csv')